# Day 4 v2 — Model 10: PhoBERT-base-v2 Improved (top-8, R-Drop, EMA 0.9999)

**Architecture:** `vinai/phobert-base-v2` (RoBERTa VN, 12L, 768d, 135M)
— Underthesea word segmentation → partial unfreeze top **8**/12 layers → mean_pooling → price head + aux category head.

**Improvements vs NB07 (session 23 — research-backed):**
| Technique | NB07 (old) | NB10 (improved) |
|---|---|---|
| keep_top_layers | 4/12 (33%) | **8/12 (67%)** — optimal for 269K samples |
| EMA decay | 0.999 (window ~1K steps) | **0.9999** (window ~10K steps) |
| warmup_ratio | 0.1 (full epoch ramp-up) | **0.05** (half epoch) |
| weight_decay | 0.02 | **0.01** (BERT standard) |
| llrd_decay | 0.9 | **0.85** (wider LR range for 8 layers) |
| batch_size | 64 | **48** (reduced for R-Drop 2x forward) |
| R-Drop | — | **alpha=0.3** (MSE consistency loss) |

**Note:** Underthesea word segmentation cache (~2-3h lần đầu) được share với NB07/08.
Cache path: `cache/phobert_seg_{train,val,test}.pkl`

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
uv add underthesea
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.phobert_model import PhoBERTRunner, PHOBERT_BASE

MODEL_NAME = PHOBERT_BASE  # "vinai/phobert-base-v2"
CACHE_DIR   = Path("cache")
WEIGHT_DIR  = Path("weights")
VAL_PRED_DIR = Path("val_predictions")

# Improved hyperparams (session 23)
KEEP_TOP      = 8       # top-8/12 layers = 67% encoder
BATCH         = 48      # reduced from 64 for R-Drop 2x forward
BASE_LR       = 2e-5
WEIGHT_DECAY  = 0.01    # was 0.02 — BERT standard
LLRD_DECAY    = 0.85    # was 0.9 — wider range for 8 layers
EPOCHS        = 10
PATIENCE      = 3
EMA_DECAY     = 0.9999  # was 0.999 — window ~10K steps
WARMUP_RATIO  = 0.05    # was 0.1 — half epoch warmup
R_DROP_ALPHA  = 0.3     # R-Drop MSE consistency loss weight

print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Underthesea Word Segmentation Cache

PhoBERT tokenizer yêu cầu text đã qua word segmentation (Underthesea).
Cache shared với NB07/NB08 — **nếu NB07 đã chạy trước, cache đã có sẵn** (~pkl files).
Nếu chưa có: tự động chạy word_segment (~2-3h trên CPU cho 269K samples).

In [ ]:
CACHE_DIR.mkdir(exist_ok=True)
SEG_TRAIN = CACHE_DIR / "phobert_seg_train.pkl"
SEG_VAL   = CACHE_DIR / "phobert_seg_val.pkl"
SEG_TEST  = CACHE_DIR / "phobert_seg_test.pkl"

print(f"Cache train: {'EXISTS' if SEG_TRAIN.exists() else 'MISSING — will compute ~2h'}")
print(f"Cache val:   {'EXISTS' if SEG_VAL.exists() else 'MISSING — will compute ~5 min'}")
print(f"Cache test:  {'EXISTS' if SEG_TEST.exists() else 'MISSING — will compute ~5 min'}")

## 3. Setup — top-8/12 layers (67% encoder unfrozen)

- Underthesea word-segment → tokenize all data
- LLRD: head lr=2e-5, mỗi layer thấp hơn × decay=0.85
- Trainable params: ~70M encoder + ~1M heads ≈ 71M total
- Expected VRAM: ~10-14GB (batch=48, top-8, R-Drop 2x forward)

In [ ]:
runner = PhoBERTRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=KEEP_TOP,
    batch_size=BATCH,
    max_length=256,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    llrd_decay=LLRD_DECAY,
    dropout=0.2,
    train_seg_cache=SEG_TRAIN,
    val_seg_cache=SEG_VAL,
)

## 4. Train

10 epochs, early stopping patience=3.
Val MAE evaluated on full 3926 samples per epoch using EMA model.
R-Drop: 2x forward per step → consistency regularization.

Expected: ~20-30 min/epoch on RTX 3090 Ti (batch=48, top-8 layers).

In [ ]:
history = runner.train(
    epochs=EPOCHS,
    patience=PATIENCE,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=EMA_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    r_drop_alpha=R_DROP_ALPHA,
)

## 5. Training History

In [ ]:
plot_training_history(history, title="PhoBERT-base Improved (top-8, R-Drop, EMA 0.9999)")

## 6. Save Weights + Val Predictions + Test Predictions

In [ ]:
WEIGHT_DIR.mkdir(exist_ok=True)
runner.save(str(WEIGHT_DIR / "phobert_base_improved.pth"))
print("Saved weights/phobert_base_improved.pth")

VAL_PRED_DIR.mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open(VAL_PRED_DIR / "phobert_base_improved_val.json", "w") as f:
    json.dump(val_preds, f)
print(f"Saved val_predictions/phobert_base_improved_val.json ({len(val_preds)} samples)")

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test, seg_cache=SEG_TEST)
with open(VAL_PRED_DIR / "phobert_base_improved_test.json", "w") as f:
    json.dump(test_preds, f)
print(f"Saved val_predictions/phobert_base_improved_test.json ({len(test_preds)} samples)")

## 7. Evaluate on 200 Test Samples

In [ ]:
def phobert_improved_pricer(item):
    return runner.inference(item)

results = evaluate(phobert_improved_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

## 8. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

ckpt = torch.load(str(WEIGHT_DIR / "phobert_base_improved.pth"), map_location="cpu", weights_only=False)
print(f"Checkpoint keys: {list(ckpt.keys())}")
print(f"keep_top_layers={ckpt['keep_top_layers']} | model_name={ckpt['model_name']}")